# Linear Projector with Bias

In [1]:
import sys
sys.path.insert(0, '..')
import torch, os
from gensim.models import KeyedVectors
from src.data import load_data
from src.models import build_model
from src.train import train
from src.evaluate import run_all_evaluations
from src.visualize import plot_curves, visualize_3d_projection
os.makedirs('../results/linear_bias', exist_ok=True)

In [2]:
embeddings = KeyedVectors.load_word2vec_format(
    '../gensim-data/glove-wiki-gigaword-300/glove-wiki-gigaword-300.gz'
)
print(f'Vocab size: {len(embeddings.key_to_index)}')

Vocab size: 400000


In [3]:
train_loader, val_loader, gender_clean, royalty_clean = load_data(embeddings)


--------------------------------------------------
  GENDER PAIRS
--------------------------------------------------
  candidates: 100 | valid ones: 100 | discarded: 0

--------------------------------------------------
  ROYALTY PAIRS
--------------------------------------------------
  candidates: 106 | valid ones: 106 | discarded: 0

  analysis: GENDER
  cos mean=0.4609 std=0.1898 | threshold 0.3: 83/100

  analysis: STATUS
  cos mean=0.2943 std=0.1196 | threshold 0.3: 51/106

  Cosine between axes: 0.3172

  Train: 106 | Val: 28


In [4]:
model = build_model('linear_bias')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model, history = train(
    projector=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, lambda_ortho=0.1, num_epochs=1500,
)
torch.save(model.state_dict(), '../results/linear_bias/model.pt')

Training:   0%|          | 0/1500 [00:00<?, ?it/s]


Best model: epoch 78, val_loss=0.0448


In [5]:
plot_curves(history)

In [6]:
metrics = run_all_evaluations(model, embeddings)
print(metrics)

Analigies:
  king - man + woman → queen  |  cos=0.9879  dist=0.2424
  prince - boy + girl → princess  |  cos=0.9933  dist=0.2513
  husband - man + woman → wife  |  cos=0.9972  dist=0.8001
  uncle - man + woman → aunt  |  cos=0.9734  dist=0.4591
  father - man + woman → mother  |  cos=0.9701  dist=0.2753
  brother - man + woman → sister  |  cos=0.9955  dist=0.2898
  king - prince + princess → queen  |  cos=0.9973  dist=0.1719
  grandfather - father + mother → grandmother  |  cos=0.9809  dist=0.2370
  actor - man + woman → actress  |  cos=0.9778  dist=0.6658
  he - man + woman → she  |  cos=0.9975  dist=0.2669
  analogy_cos=0.9871 | analogy_dist=0.3659
{'analogy_cos': 0.9870935052676414, 'analogy_dist': 0.365947425365448}


In [7]:
visualize_3d_projection(
    gender_pairs=gender_clean, royalty_pairs=royalty_clean,
    embeddings=embeddings, projector=model,
    output_html='../results/linear_bias/graph_3d.html',
    title='Linear Bias Projection - 3D Visualization',
)

Saved: ../results/linear_bias/graph_3d.html
